In [ ]:
import torch
from transformers.models.auto.tokenization_auto import AutoTokenizer
from transformers.models.auto.modeling_auto import AutoModelForMaskedLM
from transformers.utils.quantization_config import BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType
import torch.nn as nn
import bitsandbytes
from typing import *

/home/lovem/miniconda3/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Dataset Preparation

## Dataset Definition

In [2]:
MAX_LEN = 128
PAD_TOKEN_ID = -100

In [ ]:
from torch.utils.data import Dataset, random_split

class ATPBingindDataset(Dataset):
    def __init__(self, tokenizer, sequences, labels, max_length = 128):
        #vocab_size default is 128 in model config and tokenizer, but 512 has the lowest eval loss in reference paper
        self.tokenizer = tokenizer
        self.sequences = sequences
        self.labels = labels
        self.max_length = max_length

    def __len__(self)->int:
        return len(self.sequences)
    
    def __getitem__(self, index):
        try:
            sequence = self.sequences[index]
            label = self.labels[index]

            encoding = self.tokenizer(
                sequence,
                padding='max_length',
                truncation=True,
                max_length=self.max_length,
                return_tensors='pt'
            )

            padded_label = torch.full((self.max_length, ), PAD_TOKEN_ID, dtype = torch.long)
            valid_length = min(len(label), self.max_length)
            padded_label = padded_label[:valid_length]

            return {
                'input_ids': encoding['input_ids'].squeeze(0),
                'attention_mask': encoding['attention_mask'].squeeze(0),
                'labels': padded_label.clone().detach()
            }

        except Exception as e:
            print(f"Error at index {index}")
            raise e

## Split dataset function

In [4]:
def split_dataset(dataset:ATPBingindDataset, split_size = 0.8):
    train_size = int(split_size * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    return train_dataset, test_dataset

# Data preprocess

In [ ]:
import json
with open('ATP_rmsim.json') as f:
    data = json.load(f)

def generate_label(protein, positions):
    n = protein['sequence']['length']
    labels = [0 for _ in range(n)]

    for bind_site in positions:
        for i in range(bind_site[0], bind_site[1]+1):
            labels[i-1] = 1#0-index或者1-index?
    return labels

def get_binding_site(features):
    binding_site = []
    for feature in features:
        if feature['type'] == 'Binding site' and feature['ligand']['name']=='ATP':#only need FAD
            binding_site.append([feature['location']['start']['value'], feature['location']['end']['value']])
    return binding_site

In [ ]:
protein_information = []
ones_cnt = 0
zeros_cnt = 0
for protein in data['results']:
    info = {"sequence": None, "label": None}
    info['sequence'] = protein['sequence']['value']
    info['label'] = generate_label(protein, get_binding_site(protein['feature']))

# Load Model and using LoRA

In [ ]:
MODEL_ID = "../../LCPLM"

quantization_config = BitsAndBytesConfig(
    load_in_8bit = True,
    # bnb_4bit_quant_type =  "nf4",
    # bnb_4bit_use_double_quant =  True,
    # bnb_4bit_compute_dtype = torch.bfloat16
)


base_model = AutoModelForMaskedLM.from_pretrained(MODEL_ID, 
                                                  #quantization_config = quantization_config, 
                                                  trust_remote_code = True,
                                                  device_map = "auto",
                                                  torch_dtype=torch.float32
                                                 )

tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")


KeyboardInterrupt: 

In [ ]:
print(base_model) ## Check the projection layer in base model

LcPlmForMaskedLM(
  (bimamba): LcPlm(
    (backbone): BiMambaMixerModel(
      (embedding): Embedding(128, 1536)
      (layers): ModuleList(
        (0-47): 48 x Block(
          (norm): RMSNorm()
          (mixer): BiMambaWrapper(
            (mamba_fwd): Mamba(
              (in_proj): Linear(in_features=1536, out_features=6144, bias=False)
              (conv1d): Conv1d(3072, 3072, kernel_size=(4,), stride=(1,), padding=(3,), groups=3072)
              (act): SiLU()
              (x_proj): Linear(in_features=3072, out_features=128, bias=False)
              (dt_proj): Linear(in_features=96, out_features=3072, bias=True)
              (out_proj): Linear(in_features=3072, out_features=1536, bias=False)
            )
            (mamba_rev): Mamba(
              (in_proj): Linear(in_features=1536, out_features=6144, bias=False)
              (conv1d): Conv1d(3072, 3072, kernel_size=(4,), stride=(1,), padding=(3,), groups=3072)
              (act): SiLU()
              (x_proj): Linear(

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=[
        "mixer.mamba_fwd.in_proj",
        "mixer.mamba_fwd.x_proj",
        "mixer.mamba_fwd.dt_proj",
        "mixer.mamba_fwd.out_proj",
        "mixer.mamba_rev.in_proj",
        "mixer.mamba_rev.x_proj",
        "mixer.mamba_rev.dt_proj",
        "mixer.mamba_rev.out_proj"
    ],
    bias="none"
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
print("Loading model complete")
print(f"Device Type:{type(model)}")
print(f"Device:{next(model.parameters()).device}")

trainable params: 28,655,616 || all params: 1,460,725,248 || trainable%: 1.9617
Loading model complete
Device Type:<class 'peft.peft_model.PeftModelForFeatureExtraction'>
Device:cuda:0


## Define Sequence Labeling Class

In [ ]:
class LCPLMLoRAForSequenceLabeling(nn.Module):
    def __init__(self, base_model, num_labels=2):
        super().__init__()
        self.base_model = base_model
        self.classifier = nn.Linear(base_model.config.d_model, num_labels)#後續可以替換成簡單的CNN分類
        self.drop_out = nn.Dropout(0.1)
        self.loss_fn = nn.CrossEntropyLoss()
    
    def forward(self, input_ids, attention_mask = None, labels = None):
        outputs = self.base_model(
            input_ids = input_ids,
            attention_mask = attention_mask,
            output_hidden_states = True
        )
        # Get the output from basemodel
        # This part is with gradient or no gradient? It should be no gradient because the limit of GPU's memory
        sequence_output = outputs.hidden_states[-1]
        logits = self.classifier(sequence_output)
        loss = None
        if labels is not None:# 後續可使用Mask遮罩來加強訓練
            active_logits = logits.view(-1, logits.size(-1))
            active_labels = labels.view(-1)
            loss = self.loss_fn(active_logits, active_labels)

        return loss, logits

In [ ]:
sequence_model = LCPLMLoRAForSequenceLabeling(model, num_labels=2)